# 00 - environment check

Installs the `msd` wheel from our private Hugging Face workspace repo, then prints the
environment report as JSON (Python, GPU, torch/CUDA, toolkit versions, Tiny Aya access) and also
saves it as `env_check.json` in the working folder (on Kaggle, fetch that file after the run).

Needs: a secret named `HF_TOKEN` (Kaggle: Add-ons > Secrets; Colab: Secrets), internet on,
and a GPU runtime. Run it once on Kaggle and once on Colab, then copy each JSON into
`docs/runtime.md`. The report holds no secret. See `docs/runtime.md` for the one-time setup.


In [ ]:
# The private HF dataset repo that holds wheels/, scripts/, data/ and results/ (docs/runtime.md).
WORKSPACE_REPO = "YOUR-HF-USERNAME/msd-workspace"  # edit this one line once the repo exists

In [ ]:
import os


def find_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:  # Kaggle: Add-ons > Secrets, label HF_TOKEN
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:  # Colab: Secrets (key icon), name HF_TOKEN, Notebook access on
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


token = find_token()
assert token, "HF_TOKEN not found: add it as a Kaggle or Colab secret (see docs/runtime.md)"
os.environ["HF_TOKEN"] = token  # scripts and msd read it from here; it is never printed
print("HF_TOKEN found")

In [ ]:
import subprocess
import sys

from huggingface_hub import HfApi, hf_hub_download

# Keep exactly one wheel in wheels/ of the workspace repo; the last one by name is installed.
files = HfApi().list_repo_files(WORKSPACE_REPO, repo_type="dataset", token=token)
wheel = sorted(f for f in files if f.startswith("wheels/") and f.endswith(".whl"))[-1]
path = hf_hub_download(WORKSPACE_REPO, wheel, repo_type="dataset", token=token)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", path], check=True)
print("installed", wheel)

In [ ]:
import json
from pathlib import Path

from msd.envcheck import main

# main([]) on purpose: a notebook kernel's own sys.argv (-f kernel.json) must not reach the parser.
report = main([])  # prints the JSON report; HF_TOKEN is read from the environment and not shown
Path("env_check.json").write_text(json.dumps(report, indent=2), encoding="utf-8")  # the file to fetch